# Chapter 3 &mdash; `lexlt`: Dictionary Order as a Recursive Predicate

**Concept 15 of the Chapter 3 decomposition:** *Formal Lexicographic Order and `lexlt`*

Four cases, and the boundary behaviour ($\varepsilon$, prefixes) is only visible once you write it as code.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Lexlt-Predicate/Concept-Lexlt-Predicate.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Formally $s \le t$ lexicographically if $s[i]==t[i]$ up to a point and then
$s[i+1] \le t[i+1]$, with nothing beyond mattering.

```python
def lexlt(s, t):
    if (s==""): return True
    if (t==""): return False
    if (s[0] < t[0]): return True
    return (s[0] == t[0]) & lexlt(s[1::], t[1::])
```

Each of the four cases *is* part of the definition &mdash; especially what happens with
$\varepsilon$ and with prefixes.

## 2. Definitions

### The predicate from the book

In [ ]:
def lexlt(s, t):
    if (s == ""):
        return True
    if (t == ""):
        return False
    if (s[0] < t[0]):
        return True
    return (s[0] == t[0]) & lexlt(s[1::], t[1::])

### The exercise: all lexicographically-ordered pairs

In [ ]:
from itertools import product as cartesian

L1 = {"abacus","bandana","pig","cat","dodo","zulu","physics"}
L2 = {"dog","zebra","zzxyz","pimento"}

def ordered_pairs(L1, L2):
    return sorted([(a,b) for (a,b) in cartesian(L1,L2) if lexlt(a,b)])

<!-- nav-strip -->

---

&larr;&nbsp;[Ch3&nbsp;14.&nbsp;Numeric Order as a Genuine Enumeration](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Numeric-Order/Concept-Numeric-Order.ipynb) &nbsp;&middot;&nbsp; [**Chapter 3** index](https://github.com/ganeshutah/Jove/blob/master/Chapter3/README.md) &nbsp;&middot;&nbsp; [Ch3&nbsp;16.&nbsp;`nthnumeric`: Coding a Numeric-Order Generator](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Nthnumeric-Generator/Concept-Nthnumeric-Generator.ipynb)&nbsp;&rarr;

---

## 3. Tests

The four cases, one at a time.

In [ ]:
print("lexlt('', 'a')    :", lexlt('', 'a'),    " <- empty precedes everything")
print("lexlt('a', '')    :", lexlt('a', ''),    " <- nothing precedes the empty string")
print("lexlt('a', 'b')   :", lexlt('a','b'),    " <- smaller first char settles it")
print("lexlt('ab','ac')  :", lexlt('ab','ac'),  " <- equal first char: recurse")
assert lexlt('','a') and not lexlt('a','') and lexlt('a','b') and lexlt('ab','ac')

Prefix behaviour &mdash; and the deliberate quirk that `lexlt('','')` is `True`.

In [ ]:
print("lexlt('ab','abc') :", lexlt('ab','abc'), " <- a prefix precedes its extension")
print("lexlt('abc','ab') :", lexlt('abc','ab'))
print("lexlt('','')      :", lexlt('',''),      " <- True: the first case fires")
print("  so lexlt is 'less than or EQUAL' on identical strings -- note it.")

The exercise, answered.

In [ ]:
pairs = ordered_pairs(L1, L2)
print("%d ordered pairs; first 8:" % len(pairs))
for p in pairs[:8]:
    print("   ", p)
assert ('cat','dog') in pairs and ('zulu','dog') not in pairs

The same thing with a set comprehension, as the activity asks.

In [ ]:
pairs2 = sorted({(a,b) for a in L1 for b in L2 if lexlt(a,b)})
print("comprehension agrees with filter version :", pairs2 == pairs)
assert pairs2 == pairs

## 4. Exercises


1. `lexlt('','')` returns `True`. Is that "less than" or "less than or equal"?
   Change it so equal strings return `False`, and check nothing else breaks.
2. `lexlt` uses `&` rather than `and`. Does that matter here? When would it?
3. Rewrite `lexlt` iteratively and confirm it agrees on 100 random pairs.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter3/Concept-Lexlt-Predicate')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')